In [5]:
import pandas as pd
import os
import sys
import json
from typing import Optional, Dict
from tqdm.auto import tqdm
import random

from datasets import load_dataset, Dataset, DatasetDict
from datasets import Dataset, Features, Value, List

# Setup
random.seed(42)
tqdm.pandas()

from dotenv import load_dotenv
load_dotenv()

True

In [16]:
dataset = load_dataset("ShethArihant/General-Code-v1")

In [7]:
dataset["python"].features

{'task_id': Value('string'),
 'id': Value('int64'),
 'CWE_ID': Value('int64'),
 'prompt': Value('string'),
 'completion': Value('string'),
 'y_negative': Value('string'),
 'cot_steps': Value('string')}

In [8]:
dataset["python"][0]

{'task_id': 'general_code_python_0',
 'id': 0,
 'CWE_ID': -1,
 'prompt': '[{"role": "user", "content": "### Instructions:\\nThink about the problem below carefully and step-by-step. Then implement the code that meets the requirements described. Add imports if necessary.\\n\\n**Problem:**\\nPlease refactor the following coding challenge:\\n\\ndef words_without_target_advanced(s, target):\\n    \\"\\"\\"\\n    You will be provided with a string of words separated by distinct types of delimiters (such as commas, colons, semicolons, or spaces). Your task is to partition the string into words and return a dictionary. In this dictionary, keys should represent delimiters and values should be lists of the decomposed words in their original order, but the delimiters should be in reversed order throughout the string. Remove all occurrences of a specified target word. If the target word is found within any words, remove the whole word.\\n\\n    For example:\\n    words_without_target_advanced(\\"

This should be:
```python
{
    'task_id': Value('string'),
    'id': Value('string'),
    'CWE_ID': Value('int64'),
    'y_negative': Value('string'),
    'prompt': List({'content': Value('string'), 'role': Value('string')}),
    'cot_steps': Value('string'),
    'completion': List({'content': Value('string'), 'role': Value('string')})
}
```

In [9]:
ref_features = Features({
    'task_id': Value('string'),
    'id': Value('string'),
    'CWE_ID': Value('int64'),
    'y_negative': Value('string'),
    'prompt': List({'content': Value('string'), 'role': Value('string')}),
    'cot_steps': Value('string'),
    'completion': List({'content': Value('string'), 'role': Value('string')})
})

In [10]:
# OLD CODE TO MODIFY THE DATASET FEATURES

# # iterate over all splits and modify y_negative column to be an empty string if it is None
# # also convert id to string if it is an integer
# for split in dataset.keys():
#     def modify_dataset(example):
#         if example['y_negative'] is None:
#             example['y_negative'] = ""
        
#         if isinstance(example['id'], int):
#             example['id'] = str(example['id'])
#         return example

#     dataset[split] = dataset[split].map(modify_dataset)

In [ ]:
import json
from datasets import load_dataset, Dataset, DatasetDict, Features, Value, Sequence

# Load the dataset
dataset = load_dataset("ShethArihant/General-Code-v1")

# Parse JSON strings to list of dicts
def parse_messages(json_str):
    if pd.isna(json_str) or json_str == '':
        return []
    return json.loads(json_str)

In [26]:
# Convert each split
new_dataset_dict = {}
for split_name, split_data in dataset.items():
    # Convert to pandas for easier manipulation
    df = split_data.to_pandas()
    
    # Parse JSON strings
    df['prompt'] = df['prompt'].apply(parse_messages)
    df['completion'] = df['completion'].apply(parse_messages)
    
    # Convert id to string
    df['id'] = df['id'].astype(str)
    
    # Create new dataset with correct schema
    new_dataset_dict[split_name] = Dataset.from_pandas(df, features=ref_features)

In [34]:
new_dataset_dict["python"][0]

{'task_id': 'general_code_python_0',
 'id': '0',
 'CWE_ID': -1,
 'y_negative': '',
 'prompt': [{'content': '### Instructions:\nThink about the problem below carefully and step-by-step. Then implement the code that meets the requirements described. Add imports if necessary.\n\n**Problem:**\nPlease refactor the following coding challenge:\n\ndef words_without_target_advanced(s, target):\n    """\n    You will be provided with a string of words separated by distinct types of delimiters (such as commas, colons, semicolons, or spaces). Your task is to partition the string into words and return a dictionary. In this dictionary, keys should represent delimiters and values should be lists of the decomposed words in their original order, but the delimiters should be in reversed order throughout the string. Remove all occurrences of a specified target word. If the target word is found within any words, remove the whole word.\n\n    For example:\n    words_without_target_advanced("Hi, my: name is

In [35]:
for split in new_dataset_dict.keys():
    assert new_dataset_dict[split].features == ref_features

print("All checks passed!")

# Create new DatasetDict
new_dataset = DatasetDict(new_dataset_dict)

# Push back to HuggingFace
new_dataset.push_to_hub(
    "ShethArihant/General-Code-v1",
    token=os.getenv("HF_TOKEN"),
    commit_message="Fix schema: convert id to string, parse prompt/completion JSON to lists"
)

All checks passed!


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 169.75ba/s]
Processing Files (1 / 1): 100%|██████████|  820kB /  820kB,  260kB/s  
New Data Upload: 100%|██████████|  746kB /  746kB,  260kB/s  
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 145.97ba/s]
Processing Files (1 / 1): 100%|██████████|  796kB /  796kB, 18.9kB/s  
New Data Upload: 100%|██████████|  552kB /  552kB, 18.9kB/s  
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 194.44ba/s]
Processing Files (1 / 1): 100%|██████████|  819kB /  819kB,  265kB/s  
New Data Upload: 100%|██████████|  801kB /  801kB,  265kB/s  
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 147.47ba/s]
Processing Files (1 / 1): 100%|██████████|  895kB /  895kB, 73.4kB/s  
New Data Upload: 100%|██████████|  584kB /  584kB, 73.4kB/s  
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 165.21ba/s]
Processing Files (1 / 1): 100%|██████████|  845kB /  

CommitInfo(commit_url='https://huggingface.co/datasets/ShethArihant/General-Code-v1/commit/d127b979f7c67c2d8847fa3bac575bbec9f539b9', commit_message='Fix schema: convert id to string, parse prompt/completion JSON to lists', commit_description='', oid='d127b979f7c67c2d8847fa3bac575bbec9f539b9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/ShethArihant/General-Code-v1', endpoint='https://huggingface.co', repo_type='dataset', repo_id='ShethArihant/General-Code-v1'), pr_revision=None, pr_num=None)